# NumPy Mini Course
## The Engine Under Your Data Science Hood

**A Concept-First Learning Path for the Age of AI**

*Prepared for Anurag's Dojo*

---

### Learning Philosophy
> Understand the WHY → Recognize the WHAT → Let AI handle the HOW

### Priority Tags
- **🧠 CONCEPT:** Must understand deeply — your competitive advantage
- **👀 RECOGNIZE:** Know what it does when you see it — essential for evaluating AI output
- **🔧 SYNTAX:** Reference material — ask AI when needed

---
# Module 1: What Is NumPy and Why Does It Exist?
*Estimated time: 45 minutes*

## 🧠 CONCEPT: Why NumPy exists

Python lists are flexible but slow for math. They store each element as a separate object scattered across memory. NumPy arrays store numbers in a **contiguous block of memory**, just like C or Fortran arrays. This single design choice makes NumPy **10-100x faster** for numerical operations.

Think of it this way: a Python list is like books scattered across different rooms in a library. A NumPy array is like books lined up on a single shelf. When you need to read them all, one is obviously faster.

## 🧠 CONCEPT: What an ndarray is

The ndarray (N-dimensional array) is the foundation of everything in NumPy. **Every element must be the same data type.** This constraint is the source of NumPy's speed — knowing every element is, say, a 64-bit float means the computer can process them without checking types.

## 👀 RECOGNIZE: Array properties

In [ ]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6]])

print("Array:")
print(arr)
print(f"\nShape: {arr.shape}")    # (2, 3) — 2 rows, 3 columns
print(f"Ndim:  {arr.ndim}")       # 2 — number of dimensions
print(f"Dtype: {arr.dtype}")      # int64 — the data type of ALL elements
print(f"Size:  {arr.size}")       # 6 — total number of elements

### Smell Test #1

AI generates this code for your stock analysis. What's wrong?

```python
prices = np.array([100.5, 102.3, 'N/A', 105.7, 99.8])
```

**Answer:** NumPy arrays are homogeneous. The string `'N/A'` will force the entire array to become string type, breaking all math operations. You'd need to handle missing values differently (use `np.nan` or Pandas).

Let's prove it:

In [ ]:
import numpy as np

# The problem: string forces entire array to string type
prices_bad = np.array([100.5, 102.3, 'N/A', 105.7, 99.8])
print(f"Bad array dtype: {prices_bad.dtype}")  # '<U32' (string!)
# prices_bad * 2  # This would fail or give wrong results

# The fix: use np.nan for missing values
prices_good = np.array([100.5, 102.3, np.nan, 105.7, 99.8])
print(f"Good array dtype: {prices_good.dtype}")  # float64
print(f"Mean (ignoring NaN): {np.nanmean(prices_good):.2f}")

---
# Module 2: Vectorization — Thinking Without Loops
*Estimated time: 60 minutes*

## 🧠 CONCEPT: Vectorization

Vectorization means: apply an operation to an **entire array at once**, instead of looping through elements one by one. This isn't just a convenience — it's a fundamentally different way of thinking about computation.

In [ ]:
import numpy as np

# The Python way (slow):
prices_list = [100, 105, 102, 108, 110]
returns_loop = []
for i in range(1, len(prices_list)):
    returns_loop.append((prices_list[i] - prices_list[i-1]) / prices_list[i-1])
print("Loop returns:", [f"{r:.4f}" for r in returns_loop])

# The NumPy way (fast and expressive):
prices = np.array([100, 105, 102, 108, 110], dtype=float)
returns = np.diff(prices) / prices[:-1]
print("NumPy returns:", returns)

# Both produce the same result, but NumPy is 10-100x faster on large arrays

## 👀 RECOGNIZE: Common ufuncs

Ufuncs are functions that operate element-wise on arrays. When you see `np.sqrt`, `np.log`, `np.exp`, `np.sin` in AI-generated code, recognize that these operate on **every element simultaneously**.

In [ ]:
import numpy as np

arr = np.array([1, 4, 9, 16, 25])

print("Original: ", arr)
print("sqrt:     ", np.sqrt(arr))       # [1, 2, 3, 4, 5]
print("log:      ", np.log(arr))         # natural log of each element
print("exp:      ", np.exp(arr[:3]))     # e^x for first 3 elements (keeping output small)
print("arr > 10: ", arr > 10)            # [False, False, False, True, True]
# That last line is crucial — comparison operators are also vectorized!

### Smell Test #2

AI writes this. Should you accept it?

```python
result = []
for i in range(len(data)):
    result.append(np.sqrt(data[i]))
result = np.array(result)
```

**Answer:** No. This defeats the purpose of NumPy. The correct approach is simply `np.sqrt(data)`. If you see a loop over NumPy array elements in AI-generated code, that's almost always a red flag.

Let's verify:

In [ ]:
import numpy as np
import time

# Generate large array
data = np.random.default_rng(42).random(1_000_000) * 100

# BAD: Loop approach
start = time.perf_counter()
result_loop = []
for i in range(len(data)):
    result_loop.append(np.sqrt(data[i]))
result_loop = np.array(result_loop)
loop_time = time.perf_counter() - start

# GOOD: Vectorized approach
start = time.perf_counter()
result_vec = np.sqrt(data)
vec_time = time.perf_counter() - start

print(f"Loop time:       {loop_time:.4f}s")
print(f"Vectorized time: {vec_time:.6f}s")
print(f"Speedup:         {loop_time/vec_time:.0f}x")
print(f"Results match:   {np.allclose(result_loop, result_vec)}")

---
# Module 3: Indexing, Slicing, and Boolean Masking
*Estimated time: 60 minutes*

## 🧠 CONCEPT: Data selection patterns

Selecting subsets of data is something you'll do constantly. NumPy offers three complementary approaches.

## 👀 RECOGNIZE: Slice notation

In [ ]:
import numpy as np

arr = np.array([10, 20, 30, 40, 50, 60, 70])

print("Original:       ", arr)
print("arr[2]:         ", arr[2])       # 30 (single element)
print("arr[1:4]:       ", arr[1:4])     # [20, 30, 40] (start:stop, stop excluded)
print("arr[::2]:       ", arr[::2])     # [10, 30, 50, 70] (every 2nd element)
print("arr[-3:]:       ", arr[-3:])     # [50, 60, 70] (last 3 elements)

### Critical concept: NumPy slices are VIEWS, not copies

If you modify a slice, you modify the original array. This is a performance feature (no unnecessary copying) but catches people off guard.

In [ ]:
import numpy as np

arr = np.array([10, 20, 30, 40, 50, 60, 70])
print("Original arr:", arr)

# Slices are VIEWS — modifying the slice modifies the original!
subset = arr[2:5]
subset[0] = 999
print("After modifying subset[0] = 999:")
print("  arr is now:", arr)  # arr[2] is now 999!

# Use .copy() when you need independence
arr2 = np.array([10, 20, 30, 40, 50, 60, 70])
safe_copy = arr2[2:5].copy()
safe_copy[0] = 999
print("\nWith .copy():")
print("  arr2 is still:", arr2)  # Unchanged!

## 🧠 CONCEPT: Boolean masking

Boolean masking is how you filter data based on conditions. It's the NumPy equivalent of a SQL `WHERE` clause, and it's everywhere in data science.

In [ ]:
import numpy as np

temperatures = np.array([22, 35, 18, 40, 28, 15, 33])

# Create a boolean mask
hot_days = temperatures > 30
print("Temperatures:", temperatures)
print("Hot mask:    ", hot_days)        # [False, True, False, True, False, False, True]
print("Hot temps:   ", temperatures[hot_days])  # [35, 40, 33]

# Combine conditions (use & for AND, | for OR, ~ for NOT)
mild = (temperatures > 20) & (temperatures < 35)
print("\nMild temps:  ", temperatures[mild])   # [22, 28, 33]

# NOTE: Parentheses around each condition are REQUIRED
# This is a common bug that even AI sometimes gets wrong

## 👀 RECOGNIZE: Fancy indexing

In [ ]:
import numpy as np

arr = np.array([10, 20, 30, 40, 50])
indices = np.array([0, 3, 4])

print("Array:   ", arr)
print("Indices: ", indices)
print("Selected:", arr[indices])  # [10, 40, 50] — select specific positions

# KEY DIFFERENCE: Unlike slicing, fancy indexing returns a COPY, not a view
fancy = arr[indices]
fancy[0] = 999
print("\nAfter modifying fancy copy, original unchanged:", arr)

### Smell Test #3

You ask AI to "find all stock returns greater than 5%." It gives you a loop. The better approach:

In [ ]:
import numpy as np

# Generate dummy stock returns data
rng = np.random.default_rng(42)
returns = rng.normal(0.02, 0.08, 252)  # 252 trading days, mean 2%, std 8%

# BAD: Loop approach
result_loop = []
for i in range(len(returns)):
    if returns[i] > 0.05:
        result_loop.append(returns[i])
print(f"Loop found {len(result_loop)} days with >5% return")

# GOOD: Boolean masking in one line
result_mask = returns[returns > 0.05]
print(f"Mask found {len(result_mask)} days with >5% return")
print(f"Results match: {len(result_loop) == len(result_mask)}")

---
# Module 4: Shape Manipulation — Reshape, Stack, Split
*Estimated time: 45 minutes*

## 🧠 CONCEPT: Shape as data structure

In machine learning, models expect data in specific shapes. A neural network might need input shaped as `(batch_size, features)`. Understanding how to reshape without losing data is essential.

## 👀 RECOGNIZE: reshape and -1

In [ ]:
import numpy as np

arr = np.arange(12)  # [0, 1, 2, ..., 11]
print("Original:", arr, "shape:", arr.shape)

print("\nreshape(3, 4):")
print(arr.reshape(3, 4))    # 3 rows, 4 columns

print("\nreshape(2, -1):  # -1 = NumPy figures out the missing dimension")
print(arr.reshape(2, -1))   # 2 rows, NumPy calculates 6 columns

print("\nreshape(-1, 1):  # Column vector")
print(arr.reshape(-1, 1).T, "  shape:", arr.reshape(-1, 1).shape)

## 👀 RECOGNIZE: Combining arrays

In [ ]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("a:", a)
print("b:", b)
print("\nvstack:", np.vstack([a, b]))         # Vertical: [[1,2,3], [4,5,6]]
print("hstack:", np.hstack([a, b]))           # Horizontal: [1,2,3,4,5,6]
print("concatenate:", np.concatenate([a, b])) # Same as hstack for 1D

## 🧠 CONCEPT: Broadcasting rules

Broadcasting is how NumPy handles operations between arrays of different shapes. Instead of explicitly reshaping, NumPy "stretches" the smaller array to match.

**The rule:** Dimensions are compared from the right. They must either match or one of them must be 1.

In [ ]:
import numpy as np

# Subtract the mean of each column (centering data)
data = np.array([[1, 2], [3, 4], [5, 6]])  # shape (3, 2)
means = data.mean(axis=0)                    # shape (2,) → [3.0, 4.0]

print("Data (3,2):")
print(data)
print(f"\nColumn means (2,): {means}")

centered = data - means  # Broadcasting! (3,2) - (2,) works because trailing dims match
print("\nCentered (data - means):")
print(centered)
# Result: [[-2, -2], [0, 0], [2, 2]]

---
# Module 5: Aggregation and Statistical Thinking
*Estimated time: 45 minutes*

## 🧠 CONCEPT: Reduction operations

Aggregation functions "reduce" an array along an axis — collapsing many values into fewer values.

**The Axis Mental Model:** The axis you specify is the axis that **DISAPPEARS**. If you have a (3, 4) array and sum with `axis=0`, the result is shape (4,) — the first dimension (3) disappeared.

In [ ]:
import numpy as np

data = np.array([[10, 20, 30],
                 [40, 50, 60]])

print("Data:")
print(data)
print(f"Shape: {data.shape}")

print(f"\nsum() = {data.sum()}")                    # 210 (everything)
print(f"sum(axis=0) = {data.sum(axis=0)}")           # [50, 70, 90] (sum down each column)
print(f"sum(axis=1) = {data.sum(axis=1)}")           # [60, 150] (sum across each row)
print(f"mean(axis=0) = {data.mean(axis=0)}")         # Column means
print(f"std(axis=0) = {data.std(axis=0)}")           # Column standard deviations
print(f"min(axis=1) = {data.min(axis=1)}")           # Row minimums
print(f"75th pctile(axis=0) = {np.percentile(data, 75, axis=0)}")

## 🧠 CONCEPT: NumPy as statistics engine

These aggregation functions are the computational backbone of statistics.

In [ ]:
import numpy as np

# Generate dummy data for statistics examples
rng = np.random.default_rng(42)
data = rng.normal(50, 15, (100, 3))  # 100 samples, 3 features
print(f"Data shape: {data.shape}")

# Z-score normalization (you'll use this in ML constantly)
z_scores = (data - data.mean(axis=0)) / data.std(axis=0)
print(f"\nZ-score means (should be ~0):  {z_scores.mean(axis=0).round(4)}")
print(f"Z-score stds  (should be ~1):  {z_scores.std(axis=0).round(4)}")

# Correlation between two variables
stock_a = rng.normal(0.001, 0.02, 252)  # Daily returns for stock A
stock_b = stock_a * 0.6 + rng.normal(0, 0.015, 252)  # Correlated with A
corr_matrix = np.corrcoef(stock_a, stock_b)
print(f"\nCorrelation between stock A and B: {corr_matrix[0, 1]:.4f}")

---
# Module 6: Linear Algebra Essentials
*Estimated time: 45 minutes*

## 🧠 CONCEPT: Linear algebra as the language of ML

Machine learning is, at its core, linear algebra. A neural network forward pass is a sequence of matrix multiplications. PCA is eigenvalue decomposition. Linear regression is solving a system of equations.

In [ ]:
import numpy as np

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# Matrix multiplication (the @ operator is the modern way)
C = A @ B  # or np.dot(A, B)
print("A @ B =")
print(C)

# Transpose
print(f"\nA.T =\n{A.T}")  # Swap rows and columns

# Inverse (not all matrices have one!)
print(f"\nInverse of A =\n{np.linalg.inv(A)}")

# Determinant
print(f"\nDeterminant of A = {np.linalg.det(A)}")

# Eigenvalues and eigenvectors (the heart of PCA)
eigenvalues, eigenvectors = np.linalg.eig(A)
print(f"\nEigenvalues: {eigenvalues}")

### Key insight: `np.linalg.solve` vs inverse

When you see `np.linalg.inv(A) @ b`, that's a code smell — `np.linalg.solve(A, b)` is numerically more stable.

In [ ]:
import numpy as np

# Solve the system Ax = b
A = np.array([[3, 1], [1, 2]])
b = np.array([9, 8])

# BAD: Computing inverse (less stable, slower)
x_bad = np.linalg.inv(A) @ b

# GOOD: Direct solve (more stable)
x_good = np.linalg.solve(A, b)

print(f"Using inv():   x = {x_bad}")
print(f"Using solve(): x = {x_good}")
print(f"Results match: {np.allclose(x_bad, x_good)}")

# Verify: A @ x should equal b
print(f"Verification A @ x = {A @ x_good}")

---
# Module 7: Random Numbers and Simulation
*Estimated time: 30 minutes*

## 🧠 CONCEPT: Controlled randomness

Random number generation is fundamental to data science: sampling, bootstrapping, Monte Carlo simulation, train/test splits, and initializing ML model weights all depend on it. The key concept is **reproducibility** — using seeds so your "random" results can be replicated.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=42)  # Modern way, reproducible

print("5 uniform randoms [0,1):", rng.random(5).round(3))
print("Normal(0,1) samples:    ", rng.normal(0, 1, size=5).round(3))
print("Dice rolls:             ", rng.integers(1, 7, size=10))
print("Random sample (no repl):", rng.choice([1, 2, 3, 4], size=2, replace=False))

# Shuffle demo
arr = np.arange(10)
print(f"\nBefore shuffle: {arr}")
rng.shuffle(arr)  # In-place
print(f"After shuffle:  {arr}")

# NOTE: You'll see older code using np.random.seed(42) and np.random.randn().
# The modern rng = np.random.default_rng() approach is better because
# each generator is independent, avoiding global state bugs.

---
# Capstone Project: Stock Portfolio Analyzer
*Estimated time: 2-3 hours*

Build a stock portfolio analyzer using only NumPy. This ties together every module.

**Requirements:**
1. **Data Generation:** Synthetic stock data for 5 stocks over 252 days (Module 7)
2. **Returns Calculation:** Daily percentage returns via vectorization (Module 2)
3. **Statistical Analysis:** Mean returns, volatility, correlations (Module 5)
4. **Portfolio Construction:** Random weight vectors, portfolio return & risk (Module 6)
5. **Monte Carlo Simulation:** 10,000 random portfolios, find best Sharpe ratio (Modules 2, 5, 6, 7)
6. **Filtering:** Boolean masking for specific criteria (Module 3)

In [ ]:
import numpy as np

# === 1. Data Generation (Module 7) ===
rng = np.random.default_rng(42)
n_days = 252
n_stocks = 5
stock_names = ['AAPL', 'GOOG', 'MSFT', 'AMZN', 'TSLA']

# Random walk stock prices
daily_returns_raw = rng.normal(0.0005, 0.02, (n_days, n_stocks))
prices = 100 * np.cumprod(1 + daily_returns_raw, axis=0)
print(f"Price matrix shape: {prices.shape}")
print(f"Final prices: {prices[-1].round(2)}")

# === 2. Returns Calculation (Module 2) — Vectorized! ===
returns = np.diff(prices, axis=0) / prices[:-1]
print(f"\nReturns shape: {returns.shape}")

# === 3. Statistical Analysis (Module 5) ===
mean_returns = returns.mean(axis=0) * 252   # Annualized
volatilities = returns.std(axis=0) * np.sqrt(252)  # Annualized
corr_matrix = np.corrcoef(returns.T)

print("\nAnnualized Returns:", dict(zip(stock_names, mean_returns.round(4))))
print("Annualized Volatility:", dict(zip(stock_names, volatilities.round(4))))
print(f"Correlation matrix shape: {corr_matrix.shape}")

# === 4 & 5. Monte Carlo — 10,000 Random Portfolios (Modules 6, 7) ===
n_portfolios = 10000
all_weights = rng.random((n_portfolios, n_stocks))
all_weights = all_weights / all_weights.sum(axis=1, keepdims=True)  # Normalize to sum=1

# Portfolio returns and risk using matrix ops
port_returns = all_weights @ mean_returns        # (10000,) vector
cov_matrix = np.cov(returns.T) * 252
port_volatilities = np.sqrt(np.array([w @ cov_matrix @ w for w in all_weights]))

# Sharpe ratio (assuming risk-free rate = 4%)
rf = 0.04
sharpe_ratios = (port_returns - rf) / port_volatilities

# === 6. Filtering (Module 3) — Boolean masking ===
# Find portfolios with return > 10% AND volatility < 25%
good_portfolios = (port_returns > 0.10) & (port_volatilities < 0.25)
print(f"\nPortfolios with >10% return AND <25% vol: {good_portfolios.sum()}")

# Best Sharpe ratio portfolio
best_idx = np.argmax(sharpe_ratios)
print(f"\nBest Portfolio (Sharpe = {sharpe_ratios[best_idx]:.3f}):")
print(f"  Return:     {port_returns[best_idx]:.2%}")
print(f"  Volatility: {port_volatilities[best_idx]:.2%}")
print(f"  Weights:    {dict(zip(stock_names, all_weights[best_idx].round(3)))}")

---
# Quick Reference: When to Use What

| I need to... | Use this | Watch out for |
|---|---|---|
| Do math on arrays | Vectorized ops (`+`, `*`, `np.sqrt`) | Loops over elements (red flag) |
| Filter data | Boolean masking: `arr[arr > x]` | Missing parentheses in compound conditions |
| Reshape for ML | `arr.reshape(-1, 1)` or `.reshape()` | Total elements must stay same |
| Combine arrays | `np.vstack`, `np.hstack`, `np.concatenate` | Shape mismatches |
| Statistics | `np.mean`, `np.std`, `np.percentile` + axis | `axis=0` vs `axis=1` confusion |
| Matrix math | `@` operator, `np.linalg.solve` | `inv()` when `solve()` is better |
| Random sampling | `np.random.default_rng(seed=42)` | Missing seed = non-reproducible |
| Handle missing data | Switch to Pandas (`np.nan` is limited) | Strings in numeric arrays |